# Publication-Ready Chess Visualizations

This notebook creates polished visualizations for:
- LinkedIn posts
- portfolio projects
- Kaggle notebooks
- presentation-ready graphics

The focus is on:
- clarity
- storytelling
- visual hierarchy
- communication

## Imports

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sys
from pathlib import Path

## Load data

In [ ]:
PROJECT_ROOT = Path().resolve().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.config import USERNAME

DATA_PATH = (
    PROJECT_ROOT
    / "dataset"
    / "processed"
    / USERNAME
    / f"{USERNAME}_games_clean.csv"
)

PLOTS_DIR = (
    PROJECT_ROOT
    / "plots"
    / USERNAME
)

PLOTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

df = pd.read_csv(DATA_PATH)

df["date"] = pd.to_datetime(df["date"])

df['result'].unique()

## Global seaborn style

In [ ]:
sns.set_theme(
    style="ticks",
    palette="deep",
    font_scale=1.2
)

plt.rcParams.update({
    "figure.figsize": (12, 6),
    "axes.titleweight": "bold",
    "axes.titlesize": 18,
    "axes.labelsize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10
})

## Rating progression chart

### Data prep

In [ ]:
rapid = (
    df[df["time_class"] == "rapid"]
    .sort_values("date")
)

### Plot

In [ ]:
fig, ax = plt.subplots()

sns.lineplot(
    data=rapid,
    x="date",
    y="player_rating",
    linewidth=2,
    ax=ax
)

ax.set_title("Rapid Rating Progression")
ax.set_xlabel("Date")
ax.set_ylabel("Rating")

sns.despine(ax=ax)

plt.tight_layout()
plt.show()

### Export

In [ ]:
fig.savefig(
    PLOTS_DIR / "rating_progression.png",
    dpi=300,
    bbox_inches="tight"
)

## Performance by hour

### Aggregate

In [ ]:
hourly = (
    df.groupby("hour")
    .agg(
        score=("score", "mean"),
        games=("url", "count")
    )
    .reset_index()
)

### Plot

In [ ]:
best_hour = hourly.loc[hourly["score"].idxmax(), "hour"]
best_score = hourly["score"].max()

fig, ax = plt.subplots()

sns.lineplot(
    data=hourly,
    x="hour",
    y="score",
    linewidth=3,
    ax=ax
)

ax.scatter(
    best_hour,
    best_score,
    s=100,
    zorder=5
)

ax.annotate(
    f"Best hour: {best_hour}:00",
    (best_hour, best_score),
    textcoords="offset points",
    xytext=(0, 10),
    ha="center"
)

ax.set_title("When Do I Play My Best Chess?")
ax.set_xlabel("Hour of Day")
ax.set_ylabel("Average Score")
ax.set_xticks(range(24))

sns.despine(ax=ax)

plt.tight_layout()
plt.show()

### Save

In [ ]:
fig.savefig(
    PLOTS_DIR / "performance_by_hour.png",
    dpi=300,
    bbox_inches="tight"
)

## White vs Black comparison

### Data

In [ ]:
side_perf = (
    df.groupby("side")
    .agg(score=("score", "mean"))
    .reset_index()
)

### Plot

In [ ]:
fig, ax = plt.subplots()

sns.barplot(
    data=side_perf,
    x="side",
    y="score",
    palette=["#f0f0f0", "#2c2c2c"],
    edgecolor="black",
    linewidth=1.2,
    ax=ax
)

ax.set_title("White vs Black Performance")
ax.set_xlabel("Side")
ax.set_ylabel("Average Score")

sns.despine(ax=ax)

plt.tight_layout()
plt.show()

### Export

In [ ]:
fig.savefig(
    PLOTS_DIR / "white_vs_black.png",
    dpi=300,
    bbox_inches="tight"
)

## Opening analysis chart

### Prep

In [ ]:
opening_stats = (
    df.groupby("opening")
    .agg(
        score=("score", "mean"),
        games=("url", "count")
    )
    .reset_index()
)

opening_stats = (
    opening_stats[
        opening_stats["games"] >= 500
    ]
    .sort_values("score", ascending=False)
    .head(10)
)

### Plot

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))

sns.barplot(
    data=opening_stats,
    x="score",
    y="opening",
    orient="h",
    order=opening_stats["opening"],
    ax=ax
)

ax.set_title("Best Performing Openings")
ax.set_xlabel("Average Score")
ax.set_ylabel("")

sns.despine(ax=ax)

plt.tight_layout()
plt.show()

### Save

In [ ]:
fig.savefig(
    PLOTS_DIR / "best_openings.png",
    dpi=300,
    bbox_inches="tight"
)